# Clasificación SDSS: estrellas, galaxias y cuásares

Este cuaderno usa el paquete `src/` (el mismo que usa `generate_results.py`) en vez de reimplementar la lógica de datos/modelo aquí, para que el notebook y las figuras de `results/` sean consistentes entre sí.

## Objetivos
- Establecer una línea base con 5 modelos clásicos.
- Construir y evaluar una red neuronal con salida `softmax`.
- Validar con 5-fold estratificado y cuantificar incertidumbre con MC Dropout.
- Analizar importancia de características (permutación) y proyección 2D (UMAP/PCA).
- **Nota de rigor**: el dataset conserva `mjd`, `plate`, `fiberid` como features — son identificadores de la observación espectroscópica, no física del objeto, y son un riesgo conocido de fuga de metadatos (ver `scripts/run_leakage_experiment.py` y la sección 'Metadata Leakage' del README para el experimento A/B/C con K-fold).

In [ ]:
import os
import sys
import random
import numpy as np
import pandas as pd
import tensorflow as tf

REPO_ROOT = os.path.abspath('.')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.preprocessing import prepare_data
from src.models import (
    build_neural_network, compile_model, create_callbacks, train_model, train_on_fold, MCDropoutModel
)
from src.evaluation import ClassificationMetrics, ConfidenceIntervals
from src.visualization import (
    set_style, save_figure, plot_learning_curves, plot_confusion_matrix,
    plot_feature_importance, plot_uncertainty_analysis, plot_manifold_projection
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.inspection import permutation_importance

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

set_style('seaborn-v0_8-darkgrid', 'husl')
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)


## Carga y preparación de datos
Usamos `prepare_data()` de `src/preprocessing.py`: carga el CSV, valida, descarta `objid/specobjid/rerun`, separa train/test estratificado (80/20) y normaliza con `StandardScaler`. Preferimos la muestra real descargada (`data/sdss_real_sample.csv`) si existe.

In [ ]:
real_csv = 'data/sdss_real_sample.csv'
default_csv = 'data/Skyserver_SQL2_27_2018 6_51_39 PM.csv'
csv_path = real_csv if os.path.exists(real_csv) else default_csv

data = prepare_data(csv_path, test_size=0.2, random_state=SEED)
X_train = data['X_train_scaled']
X_test = data['X_test_scaled']
y_train = data['y_train']
y_test = data['y_test']
class_names = data['class_names']
feature_names = data['feature_names']

print('Features usadas:', feature_names)
print('Clases:', list(class_names))
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


## Evaluación de bases clásicas
Los mismos 5 modelos que usa `generate_results.py`, para que estos números sean directamente comparables con los de `results/model_comparison.png`.

In [ ]:
baseline_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1, max_depth=20),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=SEED, learning_rate=0.05),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=SEED),
    'KNN': KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
}

baseline_results = {}
for name, clf in baseline_models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test) if hasattr(clf, 'predict_proba') else None
    metrics = ClassificationMetrics(y_test, y_pred, y_proba, class_names)
    baseline_results[name] = metrics
    print(f"{name}: accuracy={metrics.accuracy:.4f}  macro-F1={metrics.f1_macro:.4f}")


## Red neuronal
Arquitectura y callbacks vía `src/models.py`: `dropout_rate=0.2` (bajado desde 0.4 — con ~4,000 filas de entrenamiento, dropout agresivo perjudicaba más de lo que regularizaba), `l2_reg=0.001`, y `EarlyStopping`/`ReduceLROnPlateau` monitoreando `val_accuracy` en vez de `val_loss` (con L2+dropout, `val_loss` puede subir mientras `val_accuracy` sigue mejorando).

In [ ]:
num_features = X_train.shape[1]
num_classes = len(class_names)

model = build_neural_network(num_features, num_classes, l2_reg=0.001, dropout_rate=0.2)
model = compile_model(model, learning_rate=1e-3)
model.summary()


In [ ]:
callbacks = create_callbacks(early_stopping_patience=15, reduce_lr=True, monitor='val_accuracy')
history = train_model(model, X_train, y_train, epochs=150, batch_size=32,
                      validation_split=0.2, callbacks=callbacks, verbose=1)


In [ ]:
plot_learning_curves(history, save_path=f'{RESULTS_DIR}/training_history.png')


In [ ]:
y_pred_proba = model.predict(X_test, verbose=0)
y_pred_nn = np.argmax(y_pred_proba, axis=1)
nn_metrics = ClassificationMetrics(y_test, y_pred_nn, y_pred_proba, class_names)
print(f"Accuracy red neuronal: {nn_metrics.accuracy:.4f}")
print(nn_metrics.classification_report())
plot_confusion_matrix(nn_metrics.cm, class_names, title='Matriz de confusión - Red Neuronal',
                      normalized=True, save_path=f'{RESULTS_DIR}/confusion_matrices_all_models.png')


## Validación cruzada estratificada (5-fold)
Entrenamos la misma arquitectura 5 veces sobre folds distintos para saber si el resultado es estable o depende del split particular. Usa `train_on_fold()` de `src/models.py`, pasando los mismos callbacks (`val_accuracy` + `ReduceLROnPlateau`) que usamos arriba — por defecto `train_on_fold` monitorea `val_loss` sin reducir el learning rate, así que hay que pasarlos explícitamente para ser consistentes.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_accuracies = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    model_fn = lambda: compile_model(
        build_neural_network(num_features, num_classes, l2_reg=0.001, dropout_rate=0.2)
    )
    fold_callbacks = create_callbacks(early_stopping_patience=15, reduce_lr=True, monitor='val_accuracy')
    _, _, y_fold_pred, _ = train_on_fold(
        model_fn, X_train[train_idx], y_train[train_idx], X_train[val_idx], y_train[val_idx],
        epochs=150, batch_size=32, verbose=0, callbacks=fold_callbacks
    )
    fold_acc = (y_fold_pred == y_train[val_idx]).mean()
    fold_accuracies.append(fold_acc)
    print(f'Fold {fold_idx + 1}/5: accuracy = {fold_acc:.4f}')

fold_accuracies = np.array(fold_accuracies)
mean_acc, ci_lower, ci_upper = ConfidenceIntervals.normal_ci(fold_accuracies, ci=0.95)
print(f'Media CV: {mean_acc:.4f}  (95% CI: [{ci_lower:.4f}, {ci_upper:.4f}])')


## Incertidumbre con MC Dropout
`MCDropoutModel` (en `src/models.py`) hace 50 pasadas hacia adelante con dropout activo y usa la dispersión de las predicciones como incertidumbre epistémica.

In [ ]:
mc_model = MCDropoutModel(model, n_iterations=50)
mc_mean_pred, mc_uncertainty, mc_entropy = mc_model.predict_with_uncertainty(X_test)
y_pred_mc = np.argmax(mc_mean_pred, axis=1)

max_entropy = np.log(num_classes)
mc_confidence = 1.0 - (mc_entropy / max_entropy)
mc_uncertainty_per_sample = mc_uncertainty[np.arange(len(y_pred_mc)), y_pred_mc]

uncertainty_df = pd.DataFrame({
    'confidence': mc_confidence,
    'uncertainty': mc_uncertainty_per_sample,
    'correct': (y_pred_mc == y_test)
})

print(f"MC Dropout accuracy: {(y_pred_mc == y_test).mean():.4f}  "
      f"(confianza media: {mc_confidence.mean():.4f})")
plot_uncertainty_analysis(uncertainty_df, save_path=f'{RESULTS_DIR}/uncertainty_analysis.png')


## Importancia de características
Permutación sobre un Random Forest: se mide cuánto cae el accuracy al revolver cada columna del test set. Es la pieza clave para exponer fuga de metadatos: si `plate`/`mjd`/`fiberid`/`ra` aparecen entre las más importantes, el modelo está aprendiendo la observación en vez de la física del objeto. En este dataset, `ra` (ascensión recta) resulta ser la feature más importante por un margen amplio — ver la sección 'Metadata Leakage' del README para el experimento completo de regímenes de features.

In [ ]:
rf_for_importance = RandomForestClassifier(n_estimators=300, random_state=SEED, max_depth=20, n_jobs=-1)
rf_for_importance.fit(X_train, y_train)
perm = permutation_importance(rf_for_importance, X_test, y_test, n_repeats=30, random_state=SEED, n_jobs=-1)

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)

print(importance_df.to_string(index=False))
plot_feature_importance(importance_df, top_n=10, save_path=f'{RESULTS_DIR}/feature_importance.png')


## Proyección de clases
Visualización con UMAP si está disponible, o PCA en caso contrario (no es un método de validación, solo EDA visual — un autoencoder sería una historia distinta, de representación aprendida por reconstrucción, no de preservación de vecindades locales para visualizar).

In [ ]:
try:
    from umap import UMAP
    reducer = UMAP(random_state=SEED)
    method = 'UMAP'
except ImportError:
    from sklearn.decomposition import PCA
    reducer = PCA(n_components=2, random_state=SEED)
    method = 'PCA'

X_test_proj = reducer.fit_transform(X_test)
plot_manifold_projection(X_test_proj, y_test, y_pred=y_pred_nn, class_names=class_names,
                          method=method, save_path=f'{RESULTS_DIR}/umap_projection.png')


## Comentarios de investigación
- ~~Añadir validación cruzada y estimación de incertidumbre~~ — hecho arriba (K-fold + MC Dropout).
- ~~Cuantificar fuga de metadatos con un experimento A/B/C de regímenes de features~~ — hecho en `scripts/run_leakage_experiment.py` (Fase 1 de `plan.md`), con stratified 5-fold CV por régimen. Hallazgo: Random Forest **mejora** sin metadatos (0.908→0.918), mientras que la red neuronal cae ~11 puntos y su varianza sube ~6-7x en el régimen fotométrico honesto — un efecto real pero cuya causa (información real vs. inestabilidad de optimización de esta arquitectura con menos features) no queda completamente resuelta. Ver la sección 'Metadata Leakage' del README para el detalle completo.
- Comparar con clasificación espectroscópica y fotométrica publicada.
- Usar recortes de imágenes SDSS para avanzar hacia modelos basados en imágenes.

## Descarga opcional de imágenes SDSS
Ejecute este bloque solo si desea obtener ejemplos visuales desde el servidor SDSS.

In [ ]:
import urllib.request
sdss_images_dir = f'{RESULTS_DIR}/sdss_examples'
os.makedirs(sdss_images_dir, exist_ok=True)

raw = pd.read_csv(csv_path)
for class_id, class_name in enumerate(class_names):
    mask = raw['class'] == class_name
    if mask.sum() == 0:
        continue
    idx = int(np.random.choice(np.where(mask)[0]))
    row = raw.iloc[idx]
    url = (
        'https://skyserver.sdss.org/dr18/SkyServerWS/ImgCutout/getjpeg?'
        f"ra={row['ra']}&dec={row['dec']}&scale=0.3&width=256&height=256"
    )
    filename = os.path.join(sdss_images_dir, f'{class_id:02d}_{class_name.lower()}_{idx}.jpg')
    urllib.request.urlretrieve(url, filename)
    print(f'Descargada imagen: {filename}')
